# Binding Analysis: Entity-Attribute Binding

This notebook demonstrates IRTK's `binding_analysis` module:

1. **Entity-attribute binding** – measure binding strength between positions
2. **Binding attention pattern** – find heads mediating binding
3. **Cross-position binding** – pairwise binding scores
4. **Binding through layers** – track binding across layers
5. **Multi-entity disambiguation** – how model distinguishes entities

## Why This Matters

Entity-attribute binding analysis studies how models track which properties belong to which entities in context. This is fundamental to language understanding — the model must correctly bind 'red' to 'car' and 'blue' to 'house' in complex sentences.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.binding_analysis import (
    entity_attribute_binding, binding_attention_pattern,
    cross_position_binding_score, binding_through_layers,
    multi_entity_disambiguation,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 1, 2, 3, 4, 5])
def metric_fn(logits): return float(logits[-1, 0])

In [ ]:
bind = entity_attribute_binding(model, tokens, entity_pos=3, attribute_pos=2)
print(f"Binding strength: {bind['binding_strength']:.4f}")
print(f"Attention flow: {bind['attention_flow']:.4f}")
print(f"Residual similarity: {bind['residual_similarity']:.4f}")
print(f"Layer binding: {bind['layer_binding'].round(4)}")

In [ ]:
heads = binding_attention_pattern(model, tokens, entity_pos=3, attribute_pos=2)
print(f"Top binding heads: {heads['top_binding_heads']}")
print(f"Max attention score: {heads['max_score']:.4f}")

In [ ]:
cross = cross_position_binding_score(model, tokens)
print(f"Binding matrix shape: {cross['binding_matrix'].shape}")
print(f"Strongest pair: positions {cross['strongest_pair']}")
print(f"Mean binding: {cross['mean_binding']:.4f}")

In [ ]:
layers = binding_through_layers(model, tokens, entity_pos=3, attribute_pos=2)
print(f"Layer similarities: {layers['layer_similarities'].round(4)}")
print(f"Peak layer: {layers['peak_layer']}")
print(f"Binding trend: {layers['binding_trend']}")

In [ ]:
disambig = multi_entity_disambiguation(model, tokens, entity_positions=[1, 4], query_pos=5, metric_fn=metric_fn)
print(f"Discrimination score: {disambig['discrimination_score']:.4f}")
print(f"Query->entity attention: {disambig['query_to_entity_attention'].round(4)}")
print(f"Ablation effects: {disambig['ablation_effects']}")